# TP 4.a (bis) — AI Agent **"Briefing matinal"** (Gmail + Drive + Calendar)

**Formation :** Agents IA · RAG · Orchestration LangChain & n8n
**Module :** 4 — Agents IA (Jour 3)
**Durée estimée :** 2 h 30
**Pré-requis :** TP 4.a *Research Assistant* terminé · clé Groq fonctionnelle · compte Google **dédié** (pas perso/pro).

---

## Use case

> *"Tous les matins à 8 h 30, l'agent me produit un briefing : mes 3 prochaines réunions avec leur contexte (mails récents + docs Drive liés), mes mails urgents non lus avec un brouillon de réponse, et mes deadlines à risque cette semaine."*

C'est un cas **réaliste, multi-tools, lecture + écriture** — beaucoup plus représentatif d'un agent en production que le simple Research Assistant. Il soulève les vraies questions opérationnelles : OAuth, idempotence, sandbox, gouvernance.

## Outils déclarés à l'agent

| # | Outil | API sous-jacente | Lecture/Écriture |
|---|---|---|---|
| 1 | `calendar_list_events` | Google Calendar | R |
| 2 | `gmail_search` | Gmail (DSL `is:unread`, `newer_than:1d`) | R |
| 3 | `gmail_get_thread` | Gmail | R |
| 4 | `gmail_create_draft` | Gmail | **W** (draft, jamais send) |
| 5 | `drive_search` | Drive | R |
| 6 | `drive_read_file` | Drive (export Docs/Sheets) | R |
| 7 | `rag_search_internal` | Vector store interne (stub) | R |
| 8 | `slack_post_summary` | Slack incoming webhook | **W** (DM perso) |

**Garde-fou structurel** : tout outil d'écriture crée un *draft* / DM perso — jamais d'envoi automatique. C'est la règle d'or du *human-in-the-loop*.

## Plan du notebook

| Partie | Sujet | Durée |
|---|---|---|
| 0 | Setup : OAuth Google + variables d'environnement + dépendances | 30 min |
| 1 | Implémentation des 8 outils + wrap LangChain | 30 min |
| 2 | Agent ReAct (LangGraph prebuilt) sur le golden set | 30 min |
| 3 | Limites observées de ReAct sur un briefing | 10 min |
| 4 | Workflow LangGraph `MorningBriefing` (8 nodes, reflection, retry) | 50 min |
| 5 | Comparaison & synthèse | 10 min |

---

# PARTIE 0 — Setup

## 0.1 Authentification Google — pas-à-pas (~ 15 min, une seule fois)

Google n'accepte pas une simple clé API pour Gmail / Drive / Calendar : il faut **OAuth 2.0**. Les étapes ci-dessous se font une seule fois par participant.

### Étape A — Créer un projet Google Cloud (3 min)

1. Aller sur <https://console.cloud.google.com>.
2. Bandeau du haut → **Sélectionner un projet** → **Nouveau projet**.
3. **Nom** : `tp4-briefing-agent` · **Organisation** : aucune → **Créer**.
4. Une fois créé, sélectionner ce projet dans le bandeau du haut (toujours vérifier qu'on est bien dessus).

### Étape B — Activer les 3 APIs (2 min)

Dans **APIs & Services → Bibliothèque**, chercher et **activer une par une** :

- **Gmail API**
- **Google Drive API**
- **Google Calendar API**

Bouton bleu **Activer** sur chaque page produit.

### Étape C — Configurer l'écran de consentement OAuth (3 min)

1. **APIs & Services → OAuth consent screen**.
2. **User Type** : `External` → **Créer**.
3. Champs minimaux :
   - **App name** : `TP4 Briefing Agent`
   - **User support email** : votre email
   - **Developer contact** : votre email
4. **Save and continue** → page *Scopes* : ne rien ajouter, **Save and continue**.
5. Page *Test users* : **+ Add users** → ajouter l'email du compte qui exécutera le notebook.
   ⚠️ **Sans cette étape, l'auth refusera** avec `Access blocked: app is not verified`.
6. **Save and continue** → **Back to dashboard**. Status doit être `Testing`.

### Étape D — Créer le client OAuth Desktop (2 min)

1. **APIs & Services → Credentials → + Create credentials → OAuth client ID**.
2. **Application type** : `Desktop app` (important — pas `Web app`).
3. **Name** : `tp4-notebook`.
4. **Create** → bouton **DOWNLOAD JSON**.
5. **Renommer le fichier en `credentials.json`** et le placer **dans le même dossier que ce notebook**.

### Étape E — Vérifier l'arborescence

```
.
├── tp4a-agent-morning-briefing.ipynb
├── credentials.json     ← contient client_id + client_secret (ne JAMAIS commit)
└── token.json            ← sera créé au 1er run (ne JAMAIS commit non plus)
```

Ajouter à votre `.gitignore` :
```
credentials.json
token.json
.env
```

### Étape F — Premier run (le notebook)

Quand vous exécuterez la cellule d'authentification (§ 0.4 plus bas), un onglet navigateur s'ouvrira automatiquement :

1. Choisir le compte Google à utiliser.
2. Avertissement **"Google n'a pas validé cette application"** → **Avancé → Accéder à TP4 Briefing Agent (non sécurisé)**.
   *(C'est normal en mode `Testing`. En production, on demande une *App verification* à Google.)*
3. Autoriser les **3 scopes** (lecture Gmail+drafts, lecture Calendar, lecture Drive).
4. Page de succès → fermer l'onglet. Le fichier `token.json` est créé.

Les exécutions suivantes utilisent `token.json` (refresh automatique), aucune action manuelle nécessaire.

## 0.2 Variables d'environnement

Trois variables sont attendues. Deux options pour les définir.

### Option 1 — Fichier `.env` (recommandé, propre, multi-projets)

Créer un fichier `.env` à la racine du dossier du notebook :

```
GROQ_API_KEY=gsk_xxxxxxxxxxxxxxxxxxxxxxxx
GOOGLE_CREDENTIALS_PATH=./credentials.json
SLACK_WEBHOOK_URL=https://hooks.slack.com/services/T00/B00/xxxx        # optionnel
```

Le notebook lira ce fichier via `python-dotenv` (cellule d'import plus bas).

### Option 2 — Export shell avant Jupyter

```bash
export GROQ_API_KEY="gsk_..."
export GOOGLE_CREDENTIALS_PATH="./credentials.json"
export SLACK_WEBHOOK_URL="https://hooks.slack.com/..."
jupyter notebook tp4a-agent-morning-briefing.ipynb
```

### Récupération côté Python (pattern utilisé dans tout le notebook)

```python
from dotenv import load_dotenv
load_dotenv()                                  # lit .env si présent
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "VOTRE_CLE_GROQ_ICI")
```

### Tableau récapitulatif

| Variable | Obligatoire ? | Où l'obtenir | Format attendu |
|---|---|---|---|
| `GROQ_API_KEY` | ✅ oui | <https://console.groq.com> → **API Keys** | `gsk_...` (≥ 30 caractères) |
| `GOOGLE_CREDENTIALS_PATH` | ✅ oui | étape D ci-dessus | chemin vers `credentials.json` |
| `SLACK_WEBHOOK_URL` | non (stub) | <https://api.slack.com/messaging/webhooks> | `https://hooks.slack.com/...` |

### ⚠️ Règles de sécurité

1. **Ne jamais hard-coder** une vraie clé `gsk_...` dans une cellule.
2. **Toujours** passer par `os.environ.get(..., "VOTRE_CLE_GROQ_ICI")`.
3. `credentials.json` et `token.json` sont **équivalents à un mot de passe** : accès à votre Gmail / Drive / Calendar.
4. En cas de compromission : **Google Cloud Console → Credentials → supprimer le client OAuth** (révoque tous les tokens dérivés).

## 0.3 Installation des dépendances

Décommentez la cellule suivante si vous démarrez sur un environnement vierge.

In [ ]:
# !pip install -q langchain langchain-community langchain-groq langgraph
# !pip install -q google-auth google-auth-oauthlib google-api-python-client
# !pip install -q python-dotenv requests pandas

## 0.4 Imports + chargement des variables d'environnement

Cette cellule :
1. charge `.env` si présent ;
2. lit les 3 variables ;
3. affiche un état clair (✅ / ❌) sans jamais imprimer les clés.

In [ ]:
import os
import time
import json
import base64
from datetime import datetime, timedelta, timezone
from email.mime.text import MIMEText
from typing import TypedDict, List, Dict, Any, Optional

from dotenv import load_dotenv
load_dotenv()                # lit .env si présent (silencieux sinon)

# ─── Variables d'environnement ────────────────────────────────────────────
GROQ_API_KEY            = os.environ.get("GROQ_API_KEY",            "VOTRE_CLE_GROQ_ICI")
GOOGLE_CREDENTIALS_PATH = os.environ.get("GOOGLE_CREDENTIALS_PATH", "./credentials.json")
SLACK_WEBHOOK_URL       = os.environ.get("SLACK_WEBHOOK_URL",       "")

GROQ_MODEL  = "llama-3.3-70b-versatile"
TOKEN_PATH  = "./token.json"

# ─── Vérifications ────────────────────────────────────────────────────────
GROQ_OK    = GROQ_API_KEY.startswith("gsk_") and len(GROQ_API_KEY) > 30
GCREDS_OK  = os.path.exists(GOOGLE_CREDENTIALS_PATH)
SLACK_OK   = SLACK_WEBHOOK_URL.startswith("https://hooks.slack.com/")

print(f"Groq           : {'OK' if GROQ_OK   else 'MANQUANTE'}")
print(f"Google creds   : {'OK (' + GOOGLE_CREDENTIALS_PATH + ')' if GCREDS_OK else 'MANQUANT — voir Setup 0.1 etape D'}")
print(f"Slack webhook  : {'OK' if SLACK_OK  else 'absent (stub utilise)'}")
print(f"Modele LLM     : {GROQ_MODEL}")

## 0.5 Authentification Google (1ère exécution = consentement navigateur)

À la première exécution, un onglet navigateur s'ouvre — **garder Jupyter au premier plan ensuite**, le code reprend automatiquement quand le consentement est donné. Aux exécutions suivantes, le `token.json` est réutilisé.

**Scopes demandés** (3, principe du moindre privilège) :
- `gmail.modify` — lire les mails + créer des **drafts** (jamais envoyer).
- `calendar.readonly` — lire les événements.
- `drive.readonly` — lire les fichiers.

In [ ]:
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

SCOPES = [
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/calendar.readonly",
    "https://www.googleapis.com/auth/drive.readonly",
]

def get_google_creds():
    """Charge le token.json local, le refresh si expire, sinon lance OAuth."""
    creds = None
    if os.path.exists(TOKEN_PATH):
        creds = Credentials.from_authorized_user_file(TOKEN_PATH, SCOPES)
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            if not GCREDS_OK:
                raise FileNotFoundError(
                    f"credentials.json introuvable a {GOOGLE_CREDENTIALS_PATH}. "
                    "Voir Setup 0.1 etape D."
                )
            flow = InstalledAppFlow.from_client_secrets_file(GOOGLE_CREDENTIALS_PATH, SCOPES)
            creds = flow.run_local_server(port=0)     # ouvre le navigateur
        with open(TOKEN_PATH, "w") as f:
            f.write(creds.to_json())
    return creds

if GCREDS_OK:
    creds = get_google_creds()
    gmail    = build("gmail",    "v1", credentials=creds, cache_discovery=False)
    calendar = build("calendar", "v3", credentials=creds, cache_discovery=False)
    drive    = build("drive",    "v3", credentials=creds, cache_discovery=False)
    print("Services Google prets : gmail, calendar, drive.")
else:
    gmail = calendar = drive = None
    print("Services Google indisponibles (credentials.json manquant) — les outils Google seront en stub.")

## 0.6 Sanity check : 3 appels API minimaux

Avant d'aller plus loin, on vérifie que les 3 services répondent. Si une de ces 3 cellules échoue, **arrêter ici** et corriger l'auth — tout le reste en dépend.

In [ ]:
# Calendar : 1 prochain événement
if calendar:
    now = datetime.now(timezone.utc).isoformat()
    res = calendar.events().list(
        calendarId="primary", timeMin=now, maxResults=1,
        singleEvents=True, orderBy="startTime",
    ).execute()
    items = res.get("items", [])
    if items:
        e = items[0]
        start = e["start"].get("dateTime") or e["start"].get("date")
        print(f"Prochain evt : {start} - {e.get('summary','(sans titre)')}")
    else:
        print("Aucun evt a venir.")
else:
    print("[skip] calendar service indisponible")

In [ ]:
# Gmail : combien de non-lus
if gmail:
    res = gmail.users().messages().list(userId="me", q="is:unread", maxResults=1).execute()
    total = res.get("resultSizeEstimate", 0)
    print(f"Mails non-lus (estimation) : {total}")
else:
    print("[skip] gmail service indisponible")

In [ ]:
# Drive : 3 fichiers les plus récents
if drive:
    res = drive.files().list(
        pageSize=3, orderBy="modifiedTime desc",
        fields="files(name, modifiedTime, mimeType)",
    ).execute()
    for f in res.get("files", []):
        print(f"  - {f['modifiedTime'][:10]}  {f['name']}  ({f['mimeType']})")
else:
    print("[skip] drive service indisponible")

---

# PARTIE 1 — Implémentation des 8 outils

On écrit d'abord 8 **fonctions Python pures** (testables, mockables), puis on les emballe en `Tool` LangChain pour l'agent.

## 1.1 Outils de **lecture** Calendar / Gmail / Drive

In [ ]:
def calendar_list_events(when: str = "next_3", max_results: int = 3) -> List[Dict]:
    """Liste les prochains evenements.

    when : "today", "tomorrow", "this_week", "next_3" (= 3 prochains, par defaut).
    """
    if not calendar:
        return [{"error": "calendar service unavailable"}]

    now = datetime.now(timezone.utc)
    time_min = now.isoformat()
    time_max = None
    if when == "today":
        time_max = now.replace(hour=23, minute=59, second=59).isoformat()
    elif when == "tomorrow":
        d = now + timedelta(days=1)
        time_min = d.replace(hour=0,  minute=0,  second=0).isoformat()
        time_max = d.replace(hour=23, minute=59, second=59).isoformat()
    elif when == "this_week":
        time_max = (now + timedelta(days=7)).isoformat()

    kwargs = dict(
        calendarId="primary", timeMin=time_min, maxResults=max_results,
        singleEvents=True, orderBy="startTime",
    )
    if time_max:
        kwargs["timeMax"] = time_max

    res = calendar.events().list(**kwargs).execute()
    out = []
    for e in res.get("items", []):
        out.append({
            "id":        e["id"],
            "summary":   e.get("summary", "(sans titre)"),
            "start":     e["start"].get("dateTime") or e["start"].get("date"),
            "end":       e["end"].get("dateTime")   or e["end"].get("date"),
            "attendees": [a.get("email") for a in e.get("attendees", [])],
            "description": (e.get("description") or "")[:300],
        })
    return out

In [ ]:
def gmail_search(query: str, max_results: int = 5) -> List[Dict]:
    """Recherche dans la boite (DSL Gmail : 'is:unread', 'from:x@y', 'newer_than:1d', etc.)."""
    if not gmail:
        return [{"error": "gmail service unavailable"}]

    res = gmail.users().messages().list(userId="me", q=query, maxResults=max_results).execute()
    out = []
    for m in res.get("messages", []):
        full = gmail.users().messages().get(
            userId="me", id=m["id"], format="metadata",
            metadataHeaders=["From", "Subject", "Date"],
        ).execute()
        h = {x["name"]: x["value"] for x in full["payload"].get("headers", [])}
        out.append({
            "id":        m["id"],
            "thread_id": full.get("threadId"),
            "from":      h.get("From", ""),
            "subject":   h.get("Subject", ""),
            "date":      h.get("Date", ""),
            "snippet":   full.get("snippet", ""),
        })
    return out


def gmail_get_thread(thread_id: str, max_chars: int = 1500) -> str:
    """Recupere le contenu d'un thread (concatene les snippets)."""
    if not gmail:
        return "[gmail service unavailable]"
    th = gmail.users().threads().get(userId="me", id=thread_id, format="full").execute()
    txt = "\n---\n".join(m.get("snippet", "") for m in th.get("messages", []))
    return txt[:max_chars]

In [ ]:
def drive_search(query: str, max_results: int = 5) -> List[Dict]:
    """Recherche par nom OU contenu plein-texte."""
    if not drive:
        return [{"error": "drive service unavailable"}]
    safe = query.replace("'", "\\'")
    q = f"name contains '{safe}' or fullText contains '{safe}'"
    res = drive.files().list(
        q=q, pageSize=max_results,
        fields="files(id, name, mimeType, modifiedTime, webViewLink)",
    ).execute()
    return res.get("files", [])


def drive_read_file(file_id: str, max_chars: int = 3000) -> str:
    """Lit un fichier Drive. Exporte les Google Docs/Sheets en texte."""
    if not drive:
        return "[drive service unavailable]"
    meta = drive.files().get(fileId=file_id, fields="mimeType, name").execute()
    mime = meta["mimeType"]
    try:
        if mime == "application/vnd.google-apps.document":
            data = drive.files().export(fileId=file_id, mimeType="text/plain").execute()
        elif mime == "application/vnd.google-apps.spreadsheet":
            data = drive.files().export(fileId=file_id, mimeType="text/csv").execute()
        elif mime.startswith("text/") or mime in ("application/json",):
            data = drive.files().get_media(fileId=file_id).execute()
        else:
            return f"[mimeType {mime} non supporte par drive_read_file]"
    except HttpError as e:
        return f"[erreur Drive : {e}]"
    text = data.decode("utf-8") if isinstance(data, bytes) else str(data)
    return text[:max_chars]

## 1.2 Outil d'**écriture** : `gmail_create_draft`

⚠️ **Crée un brouillon, jamais un envoi.** L'utilisateur valide manuellement avant d'envoyer.

In [ ]:
def gmail_create_draft(to: str, subject: str, body_text: str,
                       in_reply_to_thread: Optional[str] = None) -> str:
    """Cree un draft Gmail. Retourne l'ID du draft."""
    if not gmail:
        return "[gmail service unavailable]"
    msg = MIMEText(body_text)
    msg["to"]      = to
    msg["subject"] = subject
    raw = base64.urlsafe_b64encode(msg.as_bytes()).decode()
    body = {"message": {"raw": raw}}
    if in_reply_to_thread:
        body["message"]["threadId"] = in_reply_to_thread
    try:
        res = gmail.users().drafts().create(userId="me", body=body).execute()
        return res["id"]
    except HttpError as e:
        return f"[erreur draft : {e}]"

## 1.3 Outils auxiliaires : RAG interne (stub) + Slack

In [ ]:
def rag_search_internal(query: str) -> str:
    """Stub : remplacer par un Chroma sur Confluence/Notion en prod."""
    return f"[STUB rag_search_internal] Pas de KB indexee pour : '{query}'"


def slack_post_summary(text: str) -> str:
    """Poste un message via incoming webhook. Stub si SLACK_WEBHOOK_URL vide."""
    if not SLACK_OK:
        print("[STUB slack] (pas de webhook configure) :")
        print(text[:600] + ("\n..." if len(text) > 600 else ""))
        return "stub"
    import requests
    r = requests.post(SLACK_WEBHOOK_URL, json={"text": text}, timeout=10)
    return "ok" if r.status_code == 200 else f"erreur {r.status_code}"

## 1.4 Wrap LangChain — la liste `tools` que l'agent verra

Chaque `Tool` a un **nom** (action callable par le LLM) et une **description** (utilisée par le LLM pour choisir l'outil). La description est le levier #1 de qualité d'un agent — la soigner.

In [ ]:
from langchain_core.tools import Tool

# Helper : Tool a partir d'une fonction acceptant un dict json (pour les multi-args)
def _json_tool(fn, name, description):
    def _wrapped(s: str):
        try:
            kwargs = json.loads(s) if s and s.strip().startswith("{") else {"query": s}
        except json.JSONDecodeError:
            kwargs = {"query": s}
        return fn(**kwargs)
    return Tool(name=name, description=description, func=_wrapped)


tools = [
    Tool(
        name="calendar_list_events",
        description=(
            "Liste les evenements du calendrier. Argument JSON : "
            '{"when": "today|tomorrow|this_week|next_3", "max_results": 3}. '
            "Utiliser pour connaitre l'agenda de l'utilisateur."
        ),
        func=lambda s: json.dumps(calendar_list_events(**(json.loads(s) if s.strip().startswith('{') else {})), default=str)[:2500],
    ),
    Tool(
        name="gmail_search",
        description=(
            "Cherche dans Gmail avec la syntaxe Gmail : 'is:unread', 'from:x@y', "
            "'newer_than:1d', 'subject:foo'. Argument = la query string. "
            "Retourne une liste de mails (id, from, subject, snippet)."
        ),
        func=lambda q: json.dumps(gmail_search(q, max_results=5), default=str)[:2500],
    ),
    Tool(
        name="gmail_get_thread",
        description="Lit un thread Gmail entier. Argument = thread_id (obtenu via gmail_search).",
        func=lambda tid: gmail_get_thread(tid.strip()),
    ),
    Tool(
        name="gmail_create_draft",
        description=(
            "Cree un BROUILLON de mail (jamais d'envoi automatique). "
            'Argument JSON : {"to": "x@y", "subject": "...", "body_text": "...", '
            '"in_reply_to_thread": "thread_id_optionnel"}. Retourne l\'id du draft.'
        ),
        func=lambda s: gmail_create_draft(**json.loads(s)),
    ),
    Tool(
        name="drive_search",
        description="Cherche des fichiers Drive par nom ou contenu. Argument = query string.",
        func=lambda q: json.dumps(drive_search(q, max_results=5), default=str)[:2000],
    ),
    Tool(
        name="drive_read_file",
        description="Lit le contenu d'un fichier Drive. Argument = file_id.",
        func=lambda fid: drive_read_file(fid.strip()),
    ),
    Tool(
        name="rag_search_internal",
        description=(
            "Cherche dans la base de connaissances interne (Confluence/Notion). "
            "Argument = question. Stub pour ce TP."
        ),
        func=rag_search_internal,
    ),
    Tool(
        name="slack_post_summary",
        description="Poste un resume sur Slack (DM perso). Argument = texte markdown.",
        func=slack_post_summary,
    ),
]

print(f"{len(tools)} outils prets :")
for t in tools:
    print(f"  - {t.name}")

---

# PARTIE 2 — Agent ReAct (LangGraph prebuilt)

On commence par un agent **générique** : on lui pose une question, il enchaîne les outils en raisonnant. C'est le pattern *Tool Use*.

## 2.1 System prompt

In [ ]:
SYSTEM_PROMPT = """Tu es un assistant productivite personnel. Tu disposes de 8 outils :

- calendar_list_events  : agenda
- gmail_search          : recherche dans la boite mail
- gmail_get_thread      : lecture complete d'un thread
- gmail_create_draft    : creation d'un BROUILLON (jamais d'envoi)
- drive_search          : recherche de fichiers Drive
- drive_read_file       : lecture d'un fichier Drive
- rag_search_internal   : KB interne (stub pour ce TP)
- slack_post_summary    : poste sur Slack perso

REGLES :
- Lis avant d'ecrire. Avant de creer un brouillon, recupere TOUJOURS le contexte (thread complet, infos
  expediteur).
- Pour les questions hors scope (meteo, blagues, code arbitraire), refuse poliment.
- Cite tes sources : id de mail, lien Drive, id d'evenement.
- Reponds en francais.
- Pour les outils JSON multi-args, passe un objet JSON valide (ex: {\"when\": \"tomorrow\"}).
"""

print(SYSTEM_PROMPT[:300], "...")

## 2.2 Instanciation de l'agent

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model=GROQ_MODEL, api_key=GROQ_API_KEY, temperature=0)
agent_executor = create_react_agent(llm, tools, prompt=SYSTEM_PROMPT)

print(f"Agent ReAct pret avec {len(tools)} outils.")

## 2.3 Golden set — 5 demandes réalistes

| # | Demande | Outils attendus |
|---|---|---|
| D1 | Liste mes 3 prochaines réunions et résume le dernier échange avec chaque organisateur | calendar → gmail (×3) |
| D2 | Pour ma réunion `Comité Produit` (la prochaine), trouve les docs Drive partagés par les invités | calendar → drive_search |
| D3 | Quels mails non lus de la semaine méritent une réponse aujourd'hui ? Brouillon pour les 2 plus urgents | gmail_search → gmail_create_draft (×2) |
| D4 | Y a-t-il une deadline mentionnée dans mes mails des 3 derniers jours qui tombe avant vendredi ? | gmail_search + raisonnement temporel |
| D5 | Quelle est la météo à Paris demain ? | refus poli (hors périmètre) |

In [ ]:
golden_set = [
    {"id": "D1",
     "demande": "Liste mes 3 prochaines reunions et, pour chacune, resume le dernier echange "
                "Gmail avec un des invites.",
     "expected_tools": ["calendar_list_events", "gmail_search"]},
    {"id": "D2",
     "demande": "Pour ma prochaine reunion (la plus proche), trouve les fichiers Drive "
                "partages par un des invites.",
     "expected_tools": ["calendar_list_events", "drive_search"]},
    {"id": "D3",
     "demande": "Liste mes 3 mails non lus les plus recents et propose un brouillon de reponse "
                "pour le plus urgent (creer le draft).",
     "expected_tools": ["gmail_search", "gmail_get_thread", "gmail_create_draft"]},
    {"id": "D4",
     "demande": "Y a-t-il une deadline mentionnee dans mes mails recents (3 jours) qui tombe "
                "cette semaine ? Si oui, laquelle ?",
     "expected_tools": ["gmail_search"]},
    {"id": "D5",
     "demande": "Quelle est la meteo a Paris demain ?",
     "expected_tools": []},     # on attend un refus poli
]
print(f"{len(golden_set)} demandes pretes.")

## 2.4 Fonction d'exécution + traces ReAct

In [ ]:
def run_react_agent(demande: str) -> dict:
    if not GROQ_OK:
        return {"answer": "[Pas de cle Groq]", "tools_used": [], "steps": 0, "duration_s": 0}
    t0 = time.time()
    try:
        result = agent_executor.invoke(
            {"messages": [("user", demande)]},
            {"recursion_limit": 15},
        )
        duration = time.time() - t0
        messages = result["messages"]

        tools_used = []
        for msg in messages:
            for tc in getattr(msg, "tool_calls", []) or []:
                tools_used.append(tc["name"])

        final_answer = ""
        for msg in reversed(messages):
            if msg.__class__.__name__ == "AIMessage" and getattr(msg, "content", ""):
                final_answer = msg.content
                break

        # Trace facon Thought/Action/Observation
        for msg in messages:
            cls = msg.__class__.__name__
            if cls == "AIMessage":
                if msg.tool_calls:
                    for tc in msg.tool_calls:
                        args_preview = json.dumps(tc["args"])[:120]
                        print(f"  Action      : {tc['name']}({args_preview})")
                elif msg.content:
                    s = msg.content[:200]
                    print(f"  Final       : {s}{'...' if len(msg.content) > 200 else ''}")
            elif cls == "ToolMessage":
                preview = str(msg.content)[:140].replace("\n", " ")
                print(f"  Observation : [{msg.name}] {preview}...")

        return {
            "answer": final_answer,
            "tools_used": tools_used,
            "steps": len(tools_used),
            "duration_s": round(duration, 2),
        }
    except Exception as e:
        return {"answer": f"[Erreur : {e}]", "tools_used": [], "steps": 0,
                "duration_s": round(time.time() - t0, 2)}


react_results = {}
for item in golden_set:
    print("\n" + "=" * 80)
    print(f"### {item['id']} : {item['demande']}")
    print("=" * 80)
    react_results[item["id"]] = run_react_agent(item["demande"])
    r = react_results[item["id"]]
    print(f"\nOutils : {r['tools_used']} | duree {r['duration_s']}s | etapes {r['steps']}")

---

# PARTIE 3 — Limites observées de ReAct sur un briefing

Sur des demandes **isolées** (D1-D5), ReAct s'en sort plutôt bien. Mais si on lui demande **un briefing complet en une seule requête** ("fais-moi mon briefing du matin"), il décroche typiquement :

| Symptôme | Cause | Conséquence |
|---|---|---|
| Oublie de scanner l'inbox après le calendrier | Pas de plan structuré | Briefing incomplet |
| Boucle entre `gmail_search` et `gmail_get_thread` | Confusion sur quand s'arrêter | Coût × N, latence |
| Crée un draft sans avoir lu le thread | Pas d'auto-vérification | Brouillon hors-sujet |
| Pas de format markdown stable | Sortie libre | Difficile à poster sur Slack tel quel |
| Pas de retry si une étape échoue | Pas de garde-fou | Échec silencieux |

**Conclusion** : un briefing récurrent est un **workflow** — on a intérêt à fixer les étapes et à ne laisser le LLM décider qu'aux **branches conditionnelles**. C'est exactement ce que LangGraph permet en Partie 4.

---

# PARTIE 4 — Workflow LangGraph `MorningBriefing`

## 4.1 Architecture

```
   START
     |
     v
  fetch_calendar          (3 prochains evts)
     |
     v
  enrich_meetings         (pour chaque evt : mails + docs lies)
     |
     v
  scan_inbox              (mails recents non lus)
     |
     v
  classify_urgency        (LLM : urgent ? oui/non par mail)
     |
     v
  [router] urgent.count > 0 ?
     |              \
     | oui           \ non
     v                v
  draft_replies   (skip)
     |              /
     v             /
  extract_deadlines       (LLM : extraction NER + dates)
     |
     v
  assemble_briefing       (LLM : produit le markdown final)
     |
     v
  grade_briefing          (LLM : reflection — completeness check)
     |
     v
  [router] briefing_ok ?
     |              \
     | oui           \ non + iter < 1
     v                v
  post_to_slack    assemble_briefing  (retry une fois)
     |
     v
   END
```

**8 nodes · 2 edges conditionnels · 1 boucle de réflexion (max 1 retry).**

## 4.2 Définition de l'état partagé

In [ ]:
class BriefingState(TypedDict, total=False):
    today_events:    List[Dict]
    enriched_events: List[Dict]
    inbox_summary:   List[Dict]
    urgent_mails:    List[Dict]
    drafts_created:  List[Dict]
    deadlines:       List[Dict]
    briefing_md:     str
    briefing_ok:     bool
    iterations:      int
    log:             List[str]

## 4.3 Helper LLM (JSON strict)

In [ ]:
import re

def llm_json(prompt: str) -> Any:
    """Appelle le LLM et tente de parser la sortie en JSON."""
    resp = llm.invoke(prompt + "\n\nReponds UNIQUEMENT par un JSON valide, sans texte autour.")
    text = resp.content
    m = re.search(r"(\{.*\}|\[.*\])", text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group())
    except Exception:
        return None


def _log(state: BriefingState, msg: str) -> List[str]:
    return state.get("log", []) + [msg]

## 4.4 Les 8 nodes

In [ ]:
# ─── Node 1 : fetch_calendar ──────────────────────────────────────────────
def n_fetch_calendar(state: BriefingState) -> dict:
    events = calendar_list_events(when="next_3", max_results=3)
    return {
        "today_events": events,
        "log": _log(state, f"fetch_calendar: {len(events)} evts"),
    }


# ─── Node 2 : enrich_meetings ─────────────────────────────────────────────
def n_enrich_meetings(state: BriefingState) -> dict:
    enriched = []
    for evt in state.get("today_events", []):
        attendees = [a for a in evt.get("attendees", []) if a]
        related_mails, related_docs = [], []
        if attendees:
            from_q = " OR ".join(f"from:{a}" for a in attendees[:3])
            mails = gmail_search(f"({from_q}) newer_than:14d", max_results=2)
            related_mails = [{"from": m.get("from"), "subject": m.get("subject"),
                              "snippet": (m.get("snippet") or "")[:200]} for m in mails]
        title = evt.get("summary", "")
        if title and title != "(sans titre)":
            kw = title.split()[0]            # mot-cle simple
            docs = drive_search(kw, max_results=2)
            related_docs = [{"name": d.get("name"), "link": d.get("webViewLink")} for d in docs]
        enriched.append({
            "summary":       evt.get("summary"),
            "start":         evt.get("start"),
            "attendees":     attendees,
            "related_mails": related_mails,
            "related_docs":  related_docs,
        })
    return {
        "enriched_events": enriched,
        "log": _log(state, f"enrich_meetings: {len(enriched)} evts enrichis"),
    }


# ─── Node 3 : scan_inbox ──────────────────────────────────────────────────
def n_scan_inbox(state: BriefingState) -> dict:
    mails = gmail_search("is:unread newer_than:1d", max_results=8)
    return {
        "inbox_summary": mails,
        "log": _log(state, f"scan_inbox: {len(mails)} mails non lus"),
    }


# ─── Node 4 : classify_urgency ────────────────────────────────────────────
def n_classify_urgency(state: BriefingState) -> dict:
    urgent = []
    for m in state.get("inbox_summary", []):
        verdict = llm_json(
            f"Mail recu :\n"
            f"  De      : {m.get('from')}\n"
            f"  Sujet   : {m.get('subject')}\n"
            f"  Apercu  : {m.get('snippet')}\n\n"
            'Ce mail est-il URGENT (action attendue dans la journee) ? '
            'JSON : {"urgent": true|false, "reason": "..."}'
        )
        if verdict and verdict.get("urgent"):
            urgent.append({**m, "reason": verdict.get("reason", "")})
    return {
        "urgent_mails": urgent,
        "log": _log(state, f"classify_urgency: {len(urgent)} urgents"),
    }


# ─── Node 5 : draft_replies ───────────────────────────────────────────────
def n_draft_replies(state: BriefingState) -> dict:
    created = []
    for m in state.get("urgent_mails", [])[:3]:    # max 3 drafts pour eviter le spam
        thread_text = gmail_get_thread(m["thread_id"]) if m.get("thread_id") else ""
        body = llm.invoke(
            f"Tu prepares une reponse mail. Contexte du thread :\n{thread_text}\n\n"
            f"Mail recu (sujet: {m.get('subject')}) :\n{m.get('snippet')}\n\n"
            f"Redige une reponse courte, polie, en francais. Ne signe pas."
        ).content
        # Extraire l'adresse email de "Nom <email>"
        from_field = m.get("from", "")
        match = re.search(r"<([^>]+)>", from_field)
        to_addr = match.group(1) if match else from_field
        draft_id = gmail_create_draft(
            to=to_addr,
            subject=f"Re: {m.get('subject', '')}",
            body_text=body,
            in_reply_to_thread=m.get("thread_id"),
        )
        created.append({"to": to_addr, "subject": m.get("subject"), "draft_id": draft_id})
    return {
        "drafts_created": created,
        "log": _log(state, f"draft_replies: {len(created)} drafts crees"),
    }


# ─── Node 6 : extract_deadlines ───────────────────────────────────────────
def n_extract_deadlines(state: BriefingState) -> dict:
    mails = gmail_search("newer_than:3d", max_results=10)
    blob = "\n".join(f"- {m.get('subject')} | {m.get('snippet')}" for m in mails)
    out = llm_json(
        f"Aujourd'hui : {datetime.now().date().isoformat()}.\n"
        f"Mails recents :\n{blob}\n\n"
        'Extrais les deadlines explicitement mentionnees qui tombent dans les 7 prochains jours. '
        'JSON : [{"deadline": "YYYY-MM-DD", "sujet": "...", "source": "sujet du mail"}]'
    ) or []
    return {
        "deadlines": out if isinstance(out, list) else [],
        "log": _log(state, f"extract_deadlines: {len(out) if isinstance(out, list) else 0} deadlines"),
    }


# ─── Node 7 : assemble_briefing ───────────────────────────────────────────
def n_assemble_briefing(state: BriefingState) -> dict:
    payload = {
        "events":    state.get("enriched_events", []),
        "urgents":   state.get("urgent_mails", []),
        "drafts":    state.get("drafts_created", []),
        "deadlines": state.get("deadlines", []),
    }
    md_out = llm.invoke(
        "Produis un briefing markdown clair et concis pour l'utilisateur, en francais. "
        "Sections obligatoires :\n"
        "1. Reunions du jour (avec contexte)\n"
        "2. Mails urgents (mention si un draft a ete prepare)\n"
        "3. Deadlines de la semaine\n\n"
        f"Donnees JSON :\n{json.dumps(payload, default=str, ensure_ascii=False)[:4000]}\n\n"
        "Briefing :"
    ).content
    return {
        "briefing_md": md_out,
        "iterations":  state.get("iterations", 0) + 1,
        "log": _log(state, "assemble_briefing: markdown produit"),
    }


# ─── Node 8 : grade_briefing (Reflection) ─────────────────────────────────
def n_grade_briefing(state: BriefingState) -> dict:
    verdict = llm_json(
        "Voici un briefing genere par un agent :\n\n"
        f"{state.get('briefing_md', '')[:3000]}\n\n"
        "Le briefing couvre-t-il : (1) reunions du jour, (2) mails urgents, "
        "(3) deadlines ? Est-il clair et exploitable en 30 secondes de lecture ?\n"
        'JSON : {"ok": true|false, "missing": ["..."]}'
    ) or {"ok": True}
    ok = bool(verdict.get("ok", True))
    return {
        "briefing_ok": ok,
        "log": _log(state, f"grade_briefing: ok={ok}"),
    }


# ─── Node 9 : post_to_slack ───────────────────────────────────────────────
def n_post_to_slack(state: BriefingState) -> dict:
    status = slack_post_summary(state.get("briefing_md", "[briefing vide]"))
    return {"log": _log(state, f"post_to_slack: {status}")}


print("9 nodes definis (1 fetch + 1 enrich + 1 scan + 1 classify + 1 drafts + "
      "1 deadlines + 1 assemble + 1 grade + 1 post).")

## 4.5 Construction du graph

In [ ]:
from langgraph.graph import StateGraph, END

g = StateGraph(BriefingState)

g.add_node("fetch_calendar",     n_fetch_calendar)
g.add_node("enrich_meetings",    n_enrich_meetings)
g.add_node("scan_inbox",         n_scan_inbox)
g.add_node("classify_urgency",   n_classify_urgency)
g.add_node("draft_replies",      n_draft_replies)
g.add_node("extract_deadlines",  n_extract_deadlines)
g.add_node("assemble_briefing",  n_assemble_briefing)
g.add_node("grade_briefing",     n_grade_briefing)
g.add_node("post_to_slack",      n_post_to_slack)

g.set_entry_point("fetch_calendar")

# Lineaire jusqu'au routing urgents
g.add_edge("fetch_calendar",   "enrich_meetings")
g.add_edge("enrich_meetings",  "scan_inbox")
g.add_edge("scan_inbox",       "classify_urgency")

# Routing 1 : crée des drafts seulement s'il y a des mails urgents
def route_after_classify(state: BriefingState) -> str:
    return "draft_replies" if state.get("urgent_mails") else "extract_deadlines"

g.add_conditional_edges("classify_urgency", route_after_classify,
                        {"draft_replies": "draft_replies", "extract_deadlines": "extract_deadlines"})
g.add_edge("draft_replies", "extract_deadlines")

# Suite lineaire
g.add_edge("extract_deadlines", "assemble_briefing")
g.add_edge("assemble_briefing", "grade_briefing")

# Routing 2 : reflection — retry une fois si pas ok
def route_after_grade(state: BriefingState) -> str:
    if state.get("briefing_ok"):
        return "post_to_slack"
    if state.get("iterations", 0) >= 2:
        return "post_to_slack"     # on poste quand meme apres 1 retry
    return "assemble_briefing"

g.add_conditional_edges("grade_briefing", route_after_grade,
                        {"post_to_slack": "post_to_slack", "assemble_briefing": "assemble_briefing"})
g.add_edge("post_to_slack", END)

morning_briefing = g.compile()
print("Workflow MorningBriefing compile.")

## 4.6 Visualisation

In [ ]:
try:
    from IPython.display import Image, display
    png = morning_briefing.get_graph().draw_mermaid_png()
    display(Image(png))
except Exception as e:
    print(f"[viz indispo : {e}]")
    print(morning_briefing.get_graph().draw_ascii())

## 4.7 Exécution du briefing

In [ ]:
def run_morning_briefing() -> dict:
    if not GROQ_OK:
        return {"briefing_md": "[Pas de cle Groq]", "log": []}
    t0 = time.time()
    initial = {"iterations": 0, "log": []}
    try:
        final = morning_briefing.invoke(initial, {"recursion_limit": 25})
        final["duration_s"] = round(time.time() - t0, 2)
        return final
    except Exception as e:
        return {"briefing_md": f"[Erreur : {e}]", "log": [], "duration_s": round(time.time() - t0, 2)}


print("Lancement du briefing matinal...\n")
result = run_morning_briefing()
print("\n--- LOG D'EXECUTION ---")
for line in result.get("log", []):
    print("  -", line)
print(f"\nDuree totale : {result.get('duration_s')}s")

## 4.8 Affichage du briefing markdown

In [ ]:
from IPython.display import Markdown, display
display(Markdown(result.get("briefing_md", "[vide]")))

---

# PARTIE 5 — Comparaison ReAct ↔ MorningBriefing & synthèse

## 5.1 Tableau qualitatif

In [ ]:
import pandas as pd

comp = pd.DataFrame([
    {"Critere": "Plan d'execution", "ReAct": "implicite (LLM decide)", "MorningBriefing": "explicite (8 nodes)"},
    {"Critere": "Couverture",       "ReAct": "depend de la formulation", "MorningBriefing": "garantie par construction"},
    {"Critere": "Reproductibilite", "ReAct": "faible (volatilite LLM)",  "MorningBriefing": "elevee"},
    {"Critere": "Cout (appels LLM)","ReAct": "bas si converge vite",     "MorningBriefing": "plus eleve (grade nodes)"},
    {"Critere": "Auto-correction",  "ReAct": "non",                      "MorningBriefing": "oui (grade + retry x1)"},
    {"Critere": "Effets de bord (drafts)", "ReAct": "imprevisibles",     "MorningBriefing": "controles (max 3)"},
    {"Critere": "Adapte a un cron 8h30",   "ReAct": "non",               "MorningBriefing": "oui"},
])
comp

## 5.2 Patterns *Anthropic Agentic Design* couverts

| Pattern | Où dans ce notebook ? |
|---|---|
| **1. Prompt chaining** | `extract_deadlines` → `assemble_briefing` |
| **2. Parallélisation** | (séquentiel ici par simplicité — bonus : passer `enrich_meetings ‖ scan_inbox` en parallèle via `add_edge` multiple) |
| **3. Routing** | `route_after_classify` (drafts ou skip) + `route_after_grade` (post ou retry) |
| **4. Reflection loop** | `grade_briefing` → retry sur `assemble_briefing` |
| **5. Orchestrator-workers** | `enrich_meetings` (1 sous-tâche par événement) |
| **6. Tool use** | toute la couche outils (8 tools Gmail/Drive/Calendar/Slack) |

→ **Les 6 patterns en un seul scénario réaliste**.

## 5.3 Points d'attention pour la mise en production

1. **Idempotence** : ré-exécuter le briefing 2× ne doit pas créer 2× les drafts. Solution : ajouter un header `X-Briefing-Date: YYYY-MM-DD` et vérifier "draft du jour déjà créé ?" avant `gmail_create_draft`.
2. **Sandbox** : compte Google **dédié**. Jamais de compte perso/pro pendant le développement.
3. **Quota Gmail API** : 250 quota-units/utilisateur/seconde — non-bloquant en TP, à monitorer en prod (ex: Cloud Monitoring).
4. **Privacy** : ne pas logger le contenu des mails dans LangSmith ou tout outil tiers.
5. **Garde-fou écriture** : `gmail_create_draft` jamais `send`. Toute action qui envoie ou modifie réellement (delete, send) doit avoir un *human-in-the-loop* explicite.
6. **Coût LLM** : ~ 8-15 appels par briefing avec ce graph. À 0,001 $/appel sur Groq llama-3.3-70b → < 0,02 $/jour, négligeable.

## 5.4 Et après ?

- **Cron** : avec n8n (TP 4.b) ou un simple `cron` Linux + `papermill`, lancer ce notebook chaque matin à 8h30.
- **Multi-utilisateur** : extraire le `creds` par utilisateur, stocker les `token.json` dans un vault.
- **Observabilité** : activer LangSmith (cellule LangSmith du TP 4.a Research Assistant) pour tracer chaque node.
- **Ajout d'outils** : `notion_search`, `jira_create_issue`, `confluence_search` — la même architecture s'étend.

→ Suite : **`TP4-b/tp4b-n8n-morning-briefing.md`** pour reconstruire le même type d'agent en **no-code n8n**.